# பாடம் 18: கிரிப்டோகிராபிக் ரசீத்களுடன் AI முகவர்களை பாதுகாத்தல்

## கையேடு நோட்புக்

இந்த நோட்புக் நான்கு பயிற்சிகளைக் கொண்டு செல்லும்:

1. முகவர் கருவி அழைப்பிற்கு உங்கள் முதல் ரசீதைக் கையொப்பமிடவும் மற்றும் அதை சரிபார்க்கவும்.
2. ரசீதுடன் திடீர் மாற்றம் செய்து சரிபார்ப்பு தோல்வியடையும் என்பதை பாருங்கள்.
3. மூன்று ரசீத்கள் கொண்ட சங்கிலியை கட்டி, சங்கிலி முழுமையை உறுதிப்படுத்துங்கள்.
4. ஒரு Microsoft முகவர் கட்டமைப்பு கருவி அழைப்பை மூடி, ஒவ்வொரு செயலும் ரசீதைக் கொடுக்கட்டும்.

அனைத்து கிரிப்டோகிராபிக் முதன்மை கூறுகளும் நன்கு பராமரிக்கப்படும் நூலகங்களிலிருந்து இறக்குமதி செய்யப்படுகின்றன (`pynacl` Ed25519 க்கானது, `jcs` RFC 8785 வகիպற்ற JSON க்கானது, `hashlib` SHA-256 க்கானது Python தரநிலை நூலகத்திலிருந்து). ரசீதியியல் தானே நேரடியான Python ஆகும், உங்கள் வாசிப்புக்கும் திருத்தத்திற்கும் ஏற்றது.

செல்களை வரிசையாக இயக்குங்கள். ஒவ்வொரு பிரிவும் சிறியது மற்றும் சுயமறுபடியானது.


## அமைப்பு

இரண்டு சாராம்சங்களையும் நிறுவுங்கள். இரண்டும் அனுமதிப்புப் பத்திரிகைகள் உள்ளன (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## உதவிக்கருவிகள்

இந்த இரண்டு உதவிக்கருவிகள் அடிப்படை64url குறியாக்கத்தை (padding இல்லாமல்) மற்றும் ஏதாவது பொருட்களின் SHA-256 ஹாஷிங்கை கையாள்கின்றன. அவை நோட்புக் மற்ற பகுதிகளை ரசீது தர lógica மேலே கவனம் செலுத்த வைத்திருக்கின்றன.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## பிரிவு 1: உங்கள் முதல் ரசீதை கையொப்பமிடுங்கள்

**Contoso Travel** நிறுவனத்தின் எஜண்ட் சிட்னி থেকে லாஸ் ஏஞ்சலஸ் செல்லும் விமானங்களை ஒரு வாடிக்கையாளருக்காக தேடிவிட்டார் என்று கற்பனை செய்யுங்கள். இந்த கருவி அழைப்பைப் பற்றி கேட்வேயின் அறிக்கையை பதிவு செய்ய விரும்புகிறோம், இது எதிர்கால ஆய்வாளர் பக்கத்தில் வேறொரு தனித்தட்டமான வெளியிடுநர் விசையாக சரிபார்க்க முடியும்.

### படி 1.1: கையொப்ப விசையை உருவாக்கவும்

உற்பத்தியில், எஜண்டின் கையொப்ப விசை ஒரு ஹார்டுவேர் பாதுகாப்பு தொகுதியில் (HSM), Azure விசை அரணையில் அல்லது அதே போன்ற பாதுகாக்கப்பட்ட இடத்தில் இருக்கும். இந்த பாடத்திற்கு, நாங்கள் நினைவகத்தில் ஒரு புதிய விசையை உருவாக்குகிறோம்.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### படி 1.2: ரசீதின் உள்ளடக்கத்தை உருவாக்குக

உள்ளடக்கம் நாம் ரசீதால் சான்றளிக்க விரும்பும் அனைத்தையும் கொண்டுள்ளது: யார் செயல்பಟ್ಟார், எது கருவி, எந்த வாதங்களை பயன்படுத்தினார், என்ன திரும்பி வந்தது, எந்த கொள்கையின் கீழ், மற்றும் எப்போது. ரசீதானது உணர்ச்சிமிக்க உள்ளடக்கம் வெளிவிடப்படாமல் இருக்க வாதங்களையும் முடிவுகளையும் நேரடியாக சேர்க்காமல், அவற்றின் ஹேஷை எடுக்கிறது.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### படி 1.3: ரசீதை கையொப்பமிடவும் மற்றும் சேர்க்கவும்

மூன்று படிகள்:

1. ஒன்றேனும் தர்க்க ரீதியாக ஒரேபோல் இருக்கும் ரசீத்களை உருவாக்கும் இரண்டு செயலாக்கங்களும் பைட் அடையாளம் பொருந்தும் பைட்கள் உருவாக்குவதற்காக JCS பயன்படுத்தி பைலோடை நிலையான வடிவிற்கு மாற்றவும்.
2. Ed25519 தனிப்பட்ட விசையுடன் நேரடியாக நிலையான பைட்டுகளுக்கு கையொப்பமிடவும். PureEdDSA செய்தியை உள்ooksேஷனில் ஹாஷ் செய்கிறது, ஆகையால் ஒரு கூடுதல் முன்-ஹாஷ் இந்த நெறிமுறையை மாற்றும்.

பின்னர் கையொப்பம் அசல் பைலோடு இணைக்கப்பட்டு இறுதி ரசீதானது உருவாகிறது.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### படி 1.4: ரசீது சரிபார்க்கவும்

சரிபார்ப்பு செயலியை மாற்றுகிறது. நாங்கள் கையொப்பத்தை அகற்றி, canonical bytes ஐ மீண்டும் கணக்கிட்டு, கையொப்பத்தை வெளியிடுநரின் பொது விசையுடன் ஒப்பிடுகிறோம்.

சரிபார்ப்பு ஆன்லைனாக இல்லை: அழைக்க வேண்டிய சேவை ஏதும் இல்லை, மற்றும் தொடரமைப்பில் அணுகலும் இல்லை. இது இன்னும் ஒரு விஷயத்தை அானுதவிக்கு தேவையுள்ளது: வெளியிடுநரின் பொது விசை, அது ஆடிட்டர் ஏற்கனவே நம்பும் வாயிலாக கிடைக்கிறது, உதாரணமாக வெளியிடுநர் வெளியிடும் JWK தொகுப்பு அல்லது முன் நிர்ணயிக்கப்பட்ட மதிப்பு. ரசீதிலுள்ள `public_key` புலம் அந்த வேடிக்கையை செய்ய முடியாது, ஏனெனில் ரசீதினை எழுதுபவர் அந்த புலத்தையும் எழுதுகிறான். பிரிவு 2 இந்த போலியைத் தடுக்கும் முறையை காண்பிக்கிறது.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


நீங்கள் `Receipt is valid: True`, `Pre-hashed receipt valid: False` மற்றும் `Malformed receipts rejected: True` ஐக் காண வேண்டும். நேர்மறை நிலை நேரடி-JCS கையொப்ப பாதை செயல்படுவதை நிரூபிக்கிறது; எதிர்மறை கட்டுப்பாடு கையொப்ப-வட்டாரக் குறிக்கோளை ஒரு கட்டுப்பாட்டாக்கச் செய்கின்றது, அதை இலக்கியமாக bırakாமல்; கடைசி வரி தவறான உள்ளீட்டை ஒரு தவறை எழுப்பாமல் `False` ஐ திருப்புவது காட்டுகிறது.


## பிரிவு 2: ரசீதோடு குழப்பம் இடுதல்

ரசீதுகள் குழப்பம் கெடுபிடிப்பதற்கானதாகவே இருக்கும். அதை நாம் சோதிப்போம்.

ரசீதின் ஒரு மட்டுமே எழுத்தை மாற்றி சான்றிதழ் தோல்வி அடைகிறது என்பதை பார்க்கலாம்.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### என்ன நடந்தது?

நாம் `policy_id` ஐ மாற்றும்போது, canonical bytes மாறியது. கையொப்பம் (அது அசல் canonical bytesக்குள் இருந்தது) இனிமேல் பொருந்தவில்லை. சரிபார்ப்பு சரியாக `False` ஐத் திருப்புகிறது.

கையொப்பமிடப்பட்ட புலத்தை மாற்றுவது முடியாது, அதாவது `signature` பொருளின் வெளியே உள்ள எதையாவது மாற்றி, இதுவரை விலைநிலை சான்றிதழ் வெளியீட்டாளரின் பொது விசையுடன் பொருந்துமாறு சான்றிதழ் சரிபார்க்க முடியாது, மனையாட்டியாரிடம் வெளியீட்டாளரின் தனிப்பட்ட விசை இருக்காவிட்டால். `signature` பொருள் தானாக கையொப்பமிடப்படவில்லை: `sig` கையொப்பம் ஆகும், மற்றும் `verify_receipt` `public_key` ஐ புறக்கணிக்கிறது, எனவே அந்த புலத்தை மட்டும் திருத்துவதால் எதும் மாறாது. தனிப்பட்ட விசை விசை மையத்தில் இருக்கும்போதும் மற்றும் சரிபார்ப்பவர்கள் வெளியீட்டாளரின் சுயமாக வெளியிட்ட பொது விசையை பயன்படுத்தினால், கையொப்பமிடப்பட்ட புலத்தில் மாற்றம் மறைக்க இயலாது. அந்த நிலைமை முக்கியம்: அடுத்த செல் வேறு விசைக்கு சரிபார்க்கும் ஒரு திருத்தப்பட்டச் சான்றிதழை காட்டுகிறது.

உங்களால் முயற்சிக்கவும்: மேலே உள்ள செல்களில் `tool_name` அல்லது `agent_id` அல்லது `timestamp` ஐ மாற்றி மறுபடியும் இயக்கவும். கையொப்பமிடப்பட்ட புலத்தில் எந்த மாற்றமும் தவறானச் சான்றிதழை உருவாக்கும்.


### கையொப்பம் பிடிப்பதில்லை என்ற திருத்தம்

மேலே உள்ள தாக்குபவர் மீண்டும் கையொப்பமிடவில்லை. ஒரு கவனமானவர் இதைச் செய்யும்: ரசீதை மாற்றி,
தாங்களே உருவாக்கிய ஒரு விசையால் அதை கையொப்பமிடுவார், அந்த விசையை
`public_key` புலத்தில் வைக்கவும். எனவே, அதன் உள்ளடக்கத்திற்கான கையொப்பம் முழுமையாக உண்மையானதும்
ஆகும் ரசீதானது உருவாகும்.

இதை இயக்கி எந்த இரண்டு சரிபார்ப்புகளில் எது அதனை பிடிக்கிறது என்பதை பாருங்கள்.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


நீங்கள் `True` பின்னர் `False` வேண்டும் என்று காண வேண்டும். அந்த மூன்று `assert` வரிகள் அதை ஒரு
சரிபார்ப்பு ஆகும்: அந்த செல் பதட்டமானது எப்போதும் அச்சாளர் திறவுகோல் எதிர்பார்க்கும் போது ஒரு பிழையை எழுப்பும்.


பதட்டமானது உள் பொருந்தவில்லை. அதன் கையொப்பம் உண்மையில் அதன்
உள்ளடக்கங்களுக்கும் அது கொண்டிருக்கும் விசைக்கும் பொருந்துகிறது, ஆகவே விசையைக் கையேடு
வாசிக்கும் ஒரு சரிபார்ப்பாளர் அதை செல்லுபடியாக உள்ளது என அறிவிக்கிறது. ரசீது பற்றி எதுவும் தவறு போலத் தோன்றவில்லை,
ஏனென்றால் அதில் எதுவும் சீராக இல்லை.

அதனை பிடிக்கும் ஒரே விஷயம் அதிரடிக்காரர் எந்தவிதமான
பாதிப்பை ஏற்படுத்த முடியாத ஒரு விசையை எதிர்பார்த்து பரிசோதிப்பதாகும். இதுவே `verify_receipt` விசையை ஒரு argument ஆக எடுப்பதற்கான காரணம்:
`verify_receipt(receipt)` என்ற கைரேகையுடன் கூடிய ஒரு செயல்பாட்டை பாதுகாப்பாக எழுத இயலாது,
ஏனெனில் அதற்கு கிடைக்கும் ஒரே விசை அதிரடிக்காரர் வழங்கியதாக இருக்கும்.

அதே ரசீது `sample_receipts/04_resigned_receipt.json` என சேமிக்கப்படுகிறது, 
நீங்கள் அதை நோட்புக் வெளியில் பார்வையிட விரும்பினால்.


## பகுதி 3: ரசீதுகளை தொடர்களாக இணைத்தல்

ஒரு ரசீது ஒரு செயலை பாதுகாப்பதற்காகவே அமைகிறது. பெரும்பாலான முகவரிகள் பலசெயல்கள் செய்கிறார்கள். முழு தொடரையும் திருட்டு உணர்வுடையதாக மாற்ற, ஒவ்வொரு ரசீதையும் முந்தைய ரசீது ஹாஷை புதிய ரசீது தரவுகளில் சேர்ப்பதன் மூலம் முந்தைய ரசீதுடன் இணைக்கின்றோம்.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

யாராவது வழங்கப்பட்ட தொடரில் உள்ள ரசீதத்தை அகற்றினால் அல்லது மறுபடியும் வரிசைப்படுத்தினால், அடுத்த தொடர் சேதமாகும். குறுக்குவழி முடிவு அல்லது கையெழுத்தாளர் கட்டுப்பாட்டில் உள்ள பின்வரிசை மறுபிரதியெழுதுதலை கண்டறிதல் உட்பட, தற்போது பிணைக்கப்பட்ட தலை, வெளிப்புற பரிசோதனை, அல்லது உறுதிப்படுத்தப்பட்ட மக்கள் உறுதி தேவைப்படுகிறது.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


இப்போது நடுவண் ரசீதில் மாற்றம் செய்து சங்கிலியை உடைத்து மீண்டும் சரிபார்க்கவும். மாற்றிய ரசீதின் கையொப்ப சரிபார்ப்பு தோல்வி அடையுகிறது, மற்றும் அடுத்த ரசீது சங்கிலி-லிங்கு சோதனை தோல்வி அடைவதாோடு (ஏனென்றால் அதன் `previous_receipt_hash` மாற்றியுள்ள நடுவண் ரசீதின் ஹேஷுடன் பொருந்தவில்லை).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


ரசீது 0 இன்னும் சரிபார்க்கப்படுகிறது (இது மாற்றப்பட்டிருக்கவில்லை மற்றும் சார்ந்த ஒரு முன்னோடியை இல்லாதது). ரசீது 1 அதன் கையொப்ப சோதனையை தோல்வியுறுகிறது ஏனெனில் நாங்கள் `tool_args_hash` மாற்றினோம். ரசீது 2 அதன் சங்கிலி-இணைப்பு சரிபார்ப்பில் தோல்வியடைந்துள்ளது ஏனெனில் அதன் `previous_receipt_hash` மூலம் முதல் (இப்போது மாற்றிய) ரசீது 1 மீது கணக்கிடப்பட்டது.

ஒரு தாக்குதல் செய்பவர் மாற்றிய ரசீது 1 ஐ மீண்டும் கையொப்பம் வைக்கும் என்றாலும், புதிய கையொப்பம் வெளியீட்டாளரின் தனிப்பட்ட விசையை வைத்திராதவார்களுக்கு தோல்வி அடையும் (தங்களுடைய விசையால் மீண்டும் கையொப்பம் வெற்றியளிக்காது, பகுதி 2 காட்டியது போல), மற்றும் ரசீது 2 இல் சங்கிலி-இணைப்பு பொருத்தம் இல்லாமை இன்னும் மோசடியை வெளிப்படுத்தும். மாற்றத்தை மறைக்க, தாக்குதல் செய்பவர் மாற்றம் ஏற்பட்ட இடத்திலிருந்து அனைத்து ரசீதுகளையும் மீண்டும் கையொப்பம் செய்ய வேண்டும், இது வெளியீட்டாளரின் தனிப்பட்ட விசையை பெற்றிருப்பதைத் தேவைப்படும்.


## பிரிவு 4: முகவர் கருவி அழைப்பை ரசீது கையொப்பத்துடன் வளைத்தல்

ஒரு உண்மையான பரிமாற்றத்தில், ஒவ்வொரு முகவர் ஆசிரியரும் `make_receipt` அழைக்க நினைவில் வைக்க விரும்ப வேண்டாம். ஒவ்வொரு கருவி கூற்று அழைப்புக்குமான ரசீது கையொப்பம் தானாகவே அமைய வேண்டும்.

இங்கே மிக எளிமையான மாதிரி உள்ளது: எந்த Callable கருவி செயல்பாட்டையும் எடுத்துக்கொண்டு, அதற்கான ரசீது வெளியிடும் பதிப்பை திரும்ப வழங்கும் ஒரு ராப்பர் வகுப்பு. இது எந்த முகவர் கட்டமைப்புக்கும் பொருந்தும், அதில் Microsoft Agent Framework (`agent_framework.foundry`) அடிப்படையாயுள்ளது.

நீங்கள் Microsoft Foundry திட்டத்தை அமைக்கவில்லை என்றாலும்கீழுள்ள உள்ளக மாதிரி இந்த மாதிரியை காட்டுகிறது.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Microsoft Agent Framework உடன் ஒருங்கிணைத்தல்

மேலே உள்ள `ReceiptedTool` வேர்‌ப்பர் கட்டமைப்பு-தனியற்றது. Microsoft Agent Framework உடன் உருவாக்கப்பட்ட ஒரு முகவரின் உள்ளே இதைப் பயன்படுத்த, சுற்றியுள்ள செயல்பாட்டினை ஒரு கருவியாக பதிவு செய்யவும். ஒரு வரைபடம் (நீங்கள் mock ஐ உண்மையான Microsoft Foundry கருவி பதிவுச் செயல்முறையால் மாற்றுவீர்கள்):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

முகவரி கட்டமைப்பு ரசீத்கள் பற்றி எதுவும் அறியத் தேவையில்லை. ரசீதின் கையொப்பம் கருவிக்குக் சுற்றப்படுகிறது, கட்டமைப்பில் ஒட்டுவதில்லை. முகவர் மீண்டும் எழுதாமல் உள்ள முகவர் குறியீட்டிற்கு இதுவே உருவாதாரத்தைச் சேர்க்கும் வழி.


## மீளாய்வு மற்றும் நீட்டிப்பு சவால்

உங்கள் கையில்:

- Ed25519 விசைப்பலகை ஜோடியை உருவாக்கியுள்ளீர்கள்.
- ஒரு முகவர் கருவி அழைக்கு ரசீதை உருவாக்கி கையொப்பமிடியுள்ளீர்கள்.
- ரசீதையை வெளியீட்டாளரின் பொதுக் கீயைப் பயன்படுத்தி ஆன்லைன் அல்லாது சோதித்துள்ளீர்கள், அது ரசீதையிலிருந்து சுதந்திரமாக பெற்றுள்ளது.
- ரசீதையுடன் மோசடி செய்து, சோதனை தோல்வியடைந்ததை பார்த்துள்ளீர்கள்.
- மூன்று ரசீதுகளின் ஹாஷ்-செயினை உருவாக்கியுள்ளீர்கள்.
- சங்கிலியின் நடுவில் மோசடி செய்து, கையொப்ப தோல்வியும் சங்கிலி இணைப்பு தோல்வியும் இரண்டையும் பார்த்துள்ளீர்கள்.
- ஒரு கருவி செயல்பாட்டை தானாக ரசீதை கையொப்பமிடும் முறையுடன் சேந்துள்ளீர்கள்.

**நீட்டிப்பு சவால்.** ரசீத ஸ்கீமாவை `request_id` புலத்துடன் (விநியோகப்படுத்தப்பட்ட ட்ரேசிங்கிற்கான UUID) விரிவாக்கம் செய்யுங்கள். அதனை `make_receipt` இல் சேர்த்து, ரசீதைகள் இறுதிச்செயல்பாடு வரை சரியாக சோதிக்கப்படுவதை உறுதி செய்யுங்கள். பின்னர் கையொப்பமிடிய பிறகு அந்த புலத்தை மாற்றி சோதனை தோல்வியடையச் செய்யுங்கள். இது கையொப்பத்திற்கு ஒவ்வொரு பைட்டும் canonical குறியாக்கத்தின் எப்படி பங்களிக்கின்றதென்பதை உள்நோக்கமாகப் புரிந்துகொள்ள உதவும்.

**முக்கிய எல்லை.** ஒரு நிர்ணயப்பட்ட குறியீடு மற்றும் சங்கிலி சூழலில், ரசீதைகள் கையொப்ப உரிமையை, உள்ளடக்கியதின் முழுமையை மற்றும் முன்னோடியை இணைக்கும் சங்கிலியை நிரூபிக்கும். அவை செயல்பாடு, மூல உண்மை, நம்பகமான நேரம், முழுமை, கொள்கை மதிப்பீடு அல்லது சரியானதன்மை ஆகியவற்றை நிரூபிப்பதில்லை. ரசீதைகள் ஒரு அடித்தளம். ஆட்சி நீங்கள் அதிலேயே கட்டிய அமைப்பே ஆகும்.

அந்த எல்லையை மனதில் வைத்து பாடநூல் READMEயை மீண்டும் படியுங்கள். ரசீதைகளுடன் அணுகும் குழுக்களால் மிகவும் பொதுவான பிழை "நமக்கு ரசீதைகள் உள்ளன" என்பது "நாம் ஆட்சி செய்யப்பட்டு உள்ளோம்" என்று தவறாக எண்ணுவது. அது உண்மையல்ல. ரசீதைகள் முகவரின் நடத்தை கண்காணிக்கக்கூடியதாக ஆக்கும். அவை அதை சரியானதாக ஆக்குவதில்லை.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
